### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [1]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [2]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


['cases_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iii.tsv',
 'cases_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage ii.tsv',
 'cases_for_PS_TCGA-BLCA_Subtype_acinar adenocarcinoma_Stage_stage iia.tsv',
 'samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iv_case_aa14ffb4-a715-4636-8251-73d019dddd8c.tsv',
 'samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage ii.tsv',
 'samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iii.tsv',
 'subtype_for_PS_TCGA-BLCA.tsv',
 'cases_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iv.tsv',
 'samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iv.tsv',
 'stage_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma.tsv',
 'stage_for_PS_TCGA-BLCA_Subtype_acinar adenocarcinoma.tsv',
 'programs.txt',
 'rnaseq_exp_files_for_PS_TCGA-BLCA_Subtype_acinar adenocarcinoma_Stage_stage iia.tsv',
 'primary_site_program_TCGA.tsv

### All programs

In [3]:
force=False
verbose=True

prog_list = gdc.list_gdc_progams(force=force, verbose=verbose)

# prog_list

File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


### Get valid subtypes

facets = "diagnoses.primary_diagnosis"

### For each subtype → get stages

facets = "diagnoses.tumor_stage"

### For each (subtype, stage) → get samples

fields = "case_id,submitter_id,diagnoses.tumor_stage"

### Then → fetch RNA-seq files

endpoint = "/files"
field = "cases.case_id"



In [4]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

dfc.head(3)


Table opened ((33, 3)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'


,project_id,primary_site,disease_type
0,TCGA-ACC,['Adrenal gland'],['Adenomas and Adenocarcinomas']
1,TCGA-BLCA,['Bladder'],"['Epithelial Neoplasms, NOS', 'Squamous Cell N..."
2,TCGA-BRCA,['Breast'],"['Adnexal and Skin Appendage Neoplasms', 'Aden..."


### Subtypes

In [5]:
force=False
verbose=True

i=1
pid = dfc.iloc[i].project_id
primary_site = dfc.iloc[i].primary_site

print(pid, primary_site)

df_subt = gdc.build_subtypes(pid=pid, do_filter=True, force=force, verbose=verbose)

print(len(df_subt))
df_subt

TCGA-BLCA ['Bladder']
Table opened ((22, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/subtype_for_PS_TCGA-BLCA.tsv'
9


,project_id,subtype,n,subtype_raw,is_valid,frac
0,TCGA-BLCA,transitional cell carcinoma,348,transitional cell carcinoma,True,0.701613
1,TCGA-BLCA,papillary transitional cell carcinoma,76,papillary transitional cell carcinoma,True,0.153226
2,TCGA-BLCA,"papillary transitional cell carcinoma, non-inv...",66,"papillary transitional cell carcinoma, non-inv...",True,0.133065
3,TCGA-BLCA,acinar adenocarcinoma,1,acinar adenocarcinoma,True,0.002016
4,TCGA-BLCA,basaloid squamous cell carcinoma,1,basaloid squamous cell carcinoma,True,0.002016
5,TCGA-BLCA,merkel cell carcinoma,1,merkel cell carcinoma,True,0.002016
6,TCGA-BLCA,papillary renal cell carcinoma,1,papillary renal cell carcinoma,True,0.002016
7,TCGA-BLCA,"squamous cell carcinoma, clear cell type",1,"squamous cell carcinoma, clear cell type",True,0.002016
8,TCGA-BLCA,transitional cell carcinoma in situ,1,transitional cell carcinoma in situ,True,0.002016


### Stages

In [6]:
force=False
verbose=True

i=0
subtype = df_subt.iloc[i].subtype_raw

print(pid, subtype)

df_stage = gdc.build_stages(pid=pid, subtype=subtype, do_filter=True, force=force, verbose=verbose)

print(len(df_stage))
df_stage

TCGA-BLCA transitional cell carcinoma
Table opened ((14, 7)) at '/home/flavio/uv/perturb_agent/data/tcga/stage_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma.tsv'
13


,project_id,subtype,stage,n,stage_raw,is_valid,frac
0,TCGA-BLCA,transitional cell carcinoma,stage iii,125,stage iii,True,0.301205
1,TCGA-BLCA,transitional cell carcinoma,stage iv,124,stage iv,True,0.298795
2,TCGA-BLCA,transitional cell carcinoma,stage ii,113,stage ii,True,0.272289
3,TCGA-BLCA,transitional cell carcinoma,stage i,24,stage i,True,0.057831
4,TCGA-BLCA,transitional cell carcinoma,stage iia,8,stage iia,True,0.019277
5,TCGA-BLCA,transitional cell carcinoma,stage iib,8,stage iib,True,0.019277
6,TCGA-BLCA,transitional cell carcinoma,stage 0is,4,stage 0is,True,0.009639
7,TCGA-BLCA,transitional cell carcinoma,stage 0a,3,stage 0a,True,0.007229
8,TCGA-BLCA,transitional cell carcinoma,stage ia,2,stage ia,True,0.004819
9,TCGA-BLCA,transitional cell carcinoma,stage 0,1,stage 0,True,0.002410


In [7]:
force=False
verbose=True

i=0
stage = df_stage.iloc[i].stage_raw


print(pid, subtype, stage)

df_case = gdc.build_cases(pid=pid, subtype=subtype, stage=stage, force=force, verbose=verbose)
print(len(df_case))
df_case.head(12)

TCGA-BLCA transitional cell carcinoma stage iii
Table opened ((121, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iii.tsv'
121


,project_id,subtype,stage,case_id,n,frac
0,TCGA-BLCA,transitional cell carcinoma,stage iii,aa14e232-c6c6-4075-9d15-e1857373d233,1,0.008264
1,TCGA-BLCA,transitional cell carcinoma,stage iii,e44f60ec-66c4-45a2-a003-0a1faf1d6e03,1,0.008264
2,TCGA-BLCA,transitional cell carcinoma,stage iii,fc0032db-b6d8-4da3-bd1b-beec4196abf0,1,0.008264
3,TCGA-BLCA,transitional cell carcinoma,stage iii,aaa55de1-8f53-4b56-ba0a-67cff909cf81,1,0.008264
4,TCGA-BLCA,transitional cell carcinoma,stage iii,0fb043d3-d86b-4cd8-8c01-9e2b3e965bb0,1,0.008264
5,TCGA-BLCA,transitional cell carcinoma,stage iii,aff1533b-486e-40b5-a35a-cfaac5823706,1,0.008264
6,TCGA-BLCA,transitional cell carcinoma,stage iii,a0bcf1b1-4dd4-4d7f-b63e-b49296671451,1,0.008264
7,TCGA-BLCA,transitional cell carcinoma,stage iii,68adacd1-a16e-4489-9e68-792a6bd8811c,1,0.008264
8,TCGA-BLCA,transitional cell carcinoma,stage iii,b6c36037-56b1-4957-8df8-fda72b0d3c39,1,0.008264
9,TCGA-BLCA,transitional cell carcinoma,stage iii,b9b61cf9-8711-4c3a-8511-659070cfe36b,1,0.008264


### Given a PS, Subtype, and Stage --> return the SAMPLES

In [8]:
force=False
verbose=True

print(pid, subtype, stage)

df_sample = gdc.get_samples(pid=pid, subtype=subtype, stage=stage, batch_size=200, force=force, verbose=verbose)
print(len(df_sample))
df_sample.head(8)

TCGA-BLCA transitional cell carcinoma stage iii
Table opened ((367, 8)) at '/home/flavio/uv/perturb_agent/data/tcga/samples_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iii.tsv'
367


,project_id,subtype,stage,case_id,case_submitter_id,sample_id,sample_submitter_id,sample_type
0,TCGA-BLCA,transitional cell carcinoma,stage iii,7d198ee2-d80a-4759-804f-a7ef85971843,TCGA-SY-A9G5,ffd73234-26ff-4194-8086-747a33a23c81,TCGA-SY-A9G5-01Z,Primary Tumor
1,TCGA-BLCA,transitional cell carcinoma,stage iii,d63c186d-add6-492e-8250-e83f46d39d00,TCGA-DK-A3WX,fed73c89-75f1-45c9-b14b-7840c52c305f,TCGA-DK-A3WX-01A,Primary Tumor
2,TCGA-BLCA,transitional cell carcinoma,stage iii,aadcf26b-c398-489f-a86a-e8db1b5db456,TCGA-GD-A6C6,fe887339-00e4-469c-8edd-c6a11df09910,TCGA-GD-A6C6-10A,Blood Derived Normal
3,TCGA-BLCA,transitional cell carcinoma,stage iii,56d8a1b8-1268-4362-a7f1-77b76fbf1b59,TCGA-DK-A1AA,fe2914ab-8e2b-4c0a-9020-165d0009b466,TCGA-DK-A1AA-01Z,Primary Tumor
4,TCGA-BLCA,transitional cell carcinoma,stage iii,8da76432-3004-47e5-a474-bd94bd4c0b33,TCGA-FD-A5BY,fe0f8a58-7f3a-4e96-8623-a6050f690258,TCGA-FD-A5BY-01Z,Primary Tumor
5,TCGA-BLCA,transitional cell carcinoma,stage iii,493a4ff2-37a5-4b79-928d-83dbfe534556,TCGA-DK-A1AE,fde77852-768f-4949-9ccc-4faa7bdbc6f3,TCGA-DK-A1AE-10A,Blood Derived Normal
6,TCGA-BLCA,transitional cell carcinoma,stage iii,841b4582-a268-4a55-a9a2-47c7e5c3b69f,TCGA-XF-A8HE,fc7aeb8e-4258-47d0-b53b-aa4c358252ad,TCGA-XF-A8HE-01A,Primary Tumor
7,TCGA-BLCA,transitional cell carcinoma,stage iii,edc290bf-9731-4f03-bc37-64542e78e8bb,TCGA-XF-AAMX,fc18707c-4ff4-4fc4-82de-96759710951e,TCGA-XF-AAMX-01Z,Primary Tumor


In [ ]:
force=False
verbose=True

case_ids = list(np.unique(df_sample.case_id))

print(pid, subtype, stage, f"cases {len(case_ids)}")

df_files = gdc.get_expression_files_given_samples(pid=pid, subtype=subtype, stage=stage, 
                                                  case_ids=case_ids, batch_size=20, force=force, verbose=verbose)
print(len(df_files))

df_files

TCGA-BLCA transitional cell carcinoma stage iii cases 121
Searching: ........

👉 Returned 128 / Total paginated 128
Table saved ((128, 10)) at '/home/flavio/uv/perturb_agent/data/tcga/rnaseq_exp_files_for_PS_TCGA-BLCA_Subtype_transitional cell carcinoma_Stage_stage iii.tsv'
Found 128 files in selected samples.
128


,project_id,subtype,stage,case_id,sample_id,file_id,file_name,case_submitter_id,sample_submitter_id,workflow
0,TCGA-BLCA,transitional cell carcinoma,stage iii,d63c186d-add6-492e-8250-e83f46d39d00,fed73c89-75f1-45c9-b14b-7840c52c305f,b4ae8845-cb08-41d4-8a34-9ab77bebe819,4ae4a228-d935-4ea8-a3a9-82979f11d53b.rna_seq.a...,TCGA-DK-A3WX,TCGA-DK-A3WX-01A,STAR - Counts
1,TCGA-BLCA,transitional cell carcinoma,stage iii,841b4582-a268-4a55-a9a2-47c7e5c3b69f,fc7aeb8e-4258-47d0-b53b-aa4c358252ad,b6a87adc-699d-41a4-b774-6b6062de6077,32801224-d7c8-411b-9e19-febffb268bd2.rna_seq.a...,TCGA-XF-A8HE,TCGA-XF-A8HE-01A,STAR - Counts
2,TCGA-BLCA,transitional cell carcinoma,stage iii,b9b61cf9-8711-4c3a-8511-659070cfe36b,f9eb39b8-2c51-4c13-b434-d11944ed3ccc,d0d8533b-7284-49cb-96ff-f0145017084d,e0b1e359-87fc-4da5-8928-4729b1a0683a.rna_seq.a...,TCGA-XF-A8HG,TCGA-XF-A8HG-01A,STAR - Counts
3,TCGA-BLCA,transitional cell carcinoma,stage iii,9aa6522a-120a-4691-993e-3f8534f644b5,f91a5f96-cd62-4bdb-b4aa-15c24bee80de,aed471ec-47d4-4e8a-b5d7-182558c1a1d4,25b91c9a-6d52-4bc9-b6cc-e41d2c128b7b.rna_seq.a...,TCGA-K4-A3WU,TCGA-K4-A3WU-01B,STAR - Counts
4,TCGA-BLCA,transitional cell carcinoma,stage iii,fc0032db-b6d8-4da3-bd1b-beec4196abf0,f8c14183-3982-44e5-967e-03967eaa8123,a05117b2-0fd4-4f04-91b7-329a9862ce85,5c90921f-4e57-4c17-9742-4659645c448a.rna_seq.a...,TCGA-DK-A3IN,TCGA-DK-A3IN-01A,STAR - Counts
...,...,...,...,...,...,...,...,...,...,...
123,TCGA-BLCA,transitional cell carcinoma,stage iii,d17a5ab1-3789-4652-aff5-d8f4f2291f9c,085da7a1-e01d-4273-b5f2-7aab5d4d942a,683d0af9-e330-4d12-9dcd-28f0fcd30678,ec8a4bb0-6eda-4e6f-bc72-8dd20bf4247c.rna_seq.a...,TCGA-BT-A2LB,TCGA-BT-A2LB-11A,STAR - Counts
124,TCGA-BLCA,transitional cell carcinoma,stage iii,b581e799-93d1-4f1c-bcb7-7e7ae86a869e,083eb339-613b-44ad-8937-a457b158ef4c,339643c4-b114-415e-83f4-3d45cb212e9e,7ac658f5-139e-4162-9349-76ae70d81f8d.rna_seq.a...,TCGA-FD-A5BS,TCGA-FD-A5BS-01A,STAR - Counts
125,TCGA-BLCA,transitional cell carcinoma,stage iii,b027f4e2-f260-4d60-a1f4-444bc49c4a0b,072bd7a0-264f-4239-84d1-d8a85497e06f,21992c62-7349-47f8-91d0-3e55f2f8deda,4a2b75ca-922d-4ce5-b465-3932e9262bd7.rna_seq.a...,TCGA-XF-A9SM,TCGA-XF-A9SM-01A,STAR - Counts
126,TCGA-BLCA,transitional cell carcinoma,stage iii,045fe498-6a07-41ff-9956-dc0fb0483e7f,06dfea18-68e6-460d-a03f-02a92d49c171,a918f42d-99a5-4735-9542-7088d1a60743,7d054b00-f212-45b3-9791-dc8083b909e8.rna_seq.a...,TCGA-E7-A7PW,TCGA-E7-A7PW-01A,STAR - Counts


In [10]:
case_ids[:10]

['00d8e96f-f231-4475-9e9f-c1f8aec28e4e',
 '0198d8e5-a14b-44ee-b1c4-d248994845ad',
 '045fe498-6a07-41ff-9956-dc0fb0483e7f',
 '05a1402e-5a79-4d54-9430-065844ed1505',
 '06f936e9-5a90-40d3-b91a-713f2b4e6e11',
 '070c66fa-ef48-46cb-af4f-357d2c641118',
 '0f4fb205-b13c-4a85-97b3-779429e6ccdd',
 '0fb043d3-d86b-4cd8-8c01-9e2b3e965bb0',
 '1eb0adbf-dbe8-4736-97f6-2dc4ee30e943',
 '1ecfdf16-f5e9-4a08-8834-37f4b474179d']